# Notebook 17: Uni-Mol, a 3D-conformer representation

Everything we have added for the last two weeks lands 0.9-correlated with an arm we
already have, and earns no weight. The reason is that our pool is six arms over about
two and a half representations: `tab`, `trees`, `mtl13`, `mtl20` and `che` are all
downstream of one chemprop encoder. Rank 28 runs six genuinely different representations,
which is why the same tricks pay for them and not for us.

Uni-Mol is a transformer pretrained on 3D conformers. It is neither a 2D message-passing
graph nor a fingerprint, so it is the first genuinely independent view we would add.

The downstream is notebook 15's cell, unchanged: TabPFN over the embedding concatenated
with the raw qHTS and aux columns, five folds, holdout model, test model, checkpointed
per stage. Only the embedding changes, which makes this a clean comparison against the
`tabqhts` arm fitted the same way on chemprop embeddings.

**Runtime** about 1.5 to 2.5 hours, of which embedding is 20 to 40 minutes. Everything
checkpoints to Drive, so Run-all resumes after a disconnect.

**This notebook can fail at the install.** `unimol_tools` pulls its own dependencies and
downloads pretrained weights. If it does fail, cell 4 stops cleanly with a message rather
than letting a broken import cascade into a two-hour run that produces nothing.


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to a GPU before going on')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/rawq_surrogate_abd3a976a5.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
from google.colab import drive
drive.mount('/content/drive')
OUTDIR = '/content/drive/MyDrive/cyp_results'
os.makedirs(OUTDIR, exist_ok=True)
assert os.path.ismount('/content/drive'), 'Drive is not mounted; stop here, the run could not resume without it'
print('writing to', OUTDIR, '| files already there:', sorted(f for f in os.listdir(OUTDIR) if f.startswith(('nb14', 'nb15', 'part_', 'results_nb1')))[:12])

In [ ]:
!pip -q install unimol_tools==0.1.6 tabpfn==2.2.1 huggingface_hub


In [ ]:
import json, os, time, numpy as np, pandas as pd, torch
from pathlib import Path
CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', DEVICE, '| numpy', np.__version__, '| torch', torch.__version__)
print('writing to', OUT)
# If the install upgraded numpy, Colab needs a restart before the new one is live.
# Nothing is lost: the embedding checkpoints to Drive, so Run-all resumes.
if np.__version__.startswith('1.'):
    print('\n*** numpy is still 1.x after installing unimol_tools. Restart the runtime\n'
          '    (Runtime > Restart session) and Run all again. ***')


## Is Uni-Mol actually usable here

Fails loudly and early if not. Nothing below this runs until this cell prints OK.


In [ ]:
UNIMOL_OK = False
UNIMOL_ERR = None
try:
    import unimol_tools
    from unimol_tools import UniMolRepr
    UNIMOL_OK = True
    print('unimol_tools', getattr(unimol_tools, '__version__', 'unknown'))
except Exception as e:
    UNIMOL_ERR = f'{type(e).__name__}: {e}'
    print('unimol_tools unavailable ->', UNIMOL_ERR)

if UNIMOL_OK:
    try:
        _probe = UniMolRepr(data_type='molecule', remove_hs=False)
        _r = _probe.get_repr(['CCO', 'c1ccccc1O'], return_atomic_reprs=False)
        _cls = np.asarray(_r['cls_repr'] if isinstance(_r, dict) else _r, dtype=np.float32)
        print('probe ok: 2 molecules ->', _cls.shape)
        UNIMOL_DIM = int(_cls.shape[1])
    except Exception as e:
        UNIMOL_OK = False
        UNIMOL_ERR = f'{type(e).__name__}: {e}'
        print('weights or API probe failed ->', UNIMOL_ERR)

if not UNIMOL_OK:
    raise SystemExit(
        'Uni-Mol is not usable in this runtime (%s). Nothing below has run, nothing is '
        'corrupted, and no time has been wasted. Send me this message and I will either '
        'pin a working version or switch representation.' % UNIMOL_ERR)
print('\nOK. Uni-Mol embedding dimension:', UNIMOL_DIM)


## Embed the 4,905 training and 750 blind structures

Chunked, so one bad molecule costs one chunk rather than the run, with a per-molecule
retry inside a failed chunk. Failures are recorded and filled with the column mean, and
the count is printed; if it is more than a handful, the arm is not trustworthy and I
would rather see that number than have it silently imputed away.

Checkpointed to Drive. Re-running the cell resumes from the last completed chunk.


In [ ]:
import gc

EMB_PATH = OUT / 'nb17_unimol.npz'
CHUNK = 200

meta_tr = pd.read_csv(CACHE / 'meta_train.csv.gz')
meta_te = pd.read_csv(CACHE / 'meta_test.csv.gz')
SM_ALL = np.concatenate([meta_tr['SMILES'].to_numpy(), meta_te['SMILES'].to_numpy()])
N_TR, N_TE = len(meta_tr), len(meta_te)
print(f'{N_TR} training + {N_TE} blind = {len(SM_ALL)} structures to embed')

if EMB_PATH.exists():
    _z = np.load(EMB_PATH)
    E_ALL, DONE, BAD = _z['E'], set(_z['done'].tolist()), set(_z['bad'].tolist())
    print(f'resuming: {len(DONE)} of {-(-len(SM_ALL)//CHUNK)} chunks already embedded')
else:
    E_ALL = np.full((len(SM_ALL), UNIMOL_DIM), np.nan, np.float32)
    DONE, BAD = set(), set()

rep = UniMolRepr(data_type='molecule', remove_hs=False)

def embed(smis):
    r = rep.get_repr(list(smis), return_atomic_reprs=False)
    a = np.asarray(r['cls_repr'] if isinstance(r, dict) else r, dtype=np.float32)
    if a.shape[0] != len(smis):
        raise RuntimeError(f'returned {a.shape[0]} rows for {len(smis)} inputs')
    return a

t0 = time.time()
n_chunks = -(-len(SM_ALL) // CHUNK)
for ci in range(n_chunks):
    if ci in DONE:
        continue
    lo, hi = ci * CHUNK, min((ci + 1) * CHUNK, len(SM_ALL))
    sl = SM_ALL[lo:hi]
    try:
        E_ALL[lo:hi] = embed(sl)
    except Exception as e:
        print(f'  chunk {ci} failed ({type(e).__name__}); retrying one molecule at a time')
        for j, s in enumerate(sl):
            try:
                E_ALL[lo + j] = embed([s])[0]
            except Exception:
                BAD.add(lo + j)
    DONE.add(ci)
    if len(DONE) % 5 == 0 or ci == n_chunks - 1:
        np.savez_compressed(EMB_PATH, E=E_ALL, done=np.array(sorted(DONE)),
                            bad=np.array(sorted(BAD), dtype=np.int64))
        print(f'  {len(DONE)}/{n_chunks} chunks, {len(BAD)} failures ({time.time()-t0:.0f}s)',
              flush=True)
np.savez_compressed(EMB_PATH, E=E_ALL, done=np.array(sorted(DONE)),
                    bad=np.array(sorted(BAD), dtype=np.int64))
del rep; gc.collect()
try:
    torch.cuda.empty_cache()
except Exception:
    pass

nan_rows = np.flatnonzero(~np.isfinite(E_ALL).all(1))
print(f'\nembedded {len(SM_ALL)} structures, {len(BAD)} outright failures, '
      f'{len(nan_rows)} rows with any non-finite value')
if len(nan_rows):
    colmean = np.nanmean(E_ALL, 0)
    colmean = np.where(np.isfinite(colmean), colmean, 0.0)
    E_ALL[nan_rows] = colmean
    print('filled those rows with the column mean')
assert np.isfinite(E_ALL).all()
if len(nan_rows) > 50:
    print('\n*** more than 50 structures failed to embed. The arm is built on imputed '
          'rows for those and should be treated as unreliable. ***')

UNIMOL_TRAIN = E_ALL[:N_TR].copy()
UNIMOL_TEST = E_ALL[N_TR:].copy()
print('train', UNIMOL_TRAIN.shape, '| test', UNIMOL_TEST.shape,
      '| sd across dims %.4f' % float(UNIMOL_TRAIN.std(0).mean()))


## Downstream: TabPFN over the Uni-Mol embedding

Notebook 15's cell unchanged except for the embedding source, so the result is directly comparable with the `tabqhts` arm built the same way on chemprop embeddings. `REF` below is that arm's dev Spearman.


In [ ]:
import json, os, time, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from tabpfn import TabPFNRegressor

TAG = 'nb17'
PRE_PCA = True          # Uni-Mol's 512 dims go in whole; no-PCA measured +0.0125 for us
CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
ISO = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
REF = {'CYP1A2': 0.5610, 'CYP2C9': 0.7189, 'CYP2D6': 0.4219, 'CYP3A4': 0.8074}   # tabqhts dev Spearman
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
meta = meta_tr; hold = np.load(CACHE / 'holdout_mask.npy')
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
EMB = UNIMOL_TRAIN.astype(np.float32); EMB_TE = UNIMOL_TEST.astype(np.float32)
RAW = np.hstack([np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'aux_train.npy')]).astype(np.float32)
RAW_TE = np.hstack([np.load(CACHE / 'qhts_te.npy'), np.load(CACHE / 'aux_test.npy')]).astype(np.float32)
print('device', DEVICE, '| emb', EMB.shape, '| raw qhts+aux', RAW.shape, '| test', EMB_TE.shape)
assert len(EMB) == len(meta) == len(RAW) and len(EMB_TE) == len(RAW_TE)

def feats(tr_idx, ev_X, ev_RAW):
    """Fit the transform on training rows only; return (train features, eval features)."""
    a, b = EMB[tr_idx], ev_X
    if not PRE_PCA:
        n = min(200, a.shape[0], a.shape[1])
        imp = SimpleImputer(strategy='median').fit(a); sc = StandardScaler().fit(imp.transform(a))
        pca = PCA(n_components=n, random_state=42).fit(sc.transform(imp.transform(a)))
        f = lambda m: pca.transform(sc.transform(imp.transform(m))).astype(np.float32)
        a, b = f(a), f(b)
    ri = SimpleImputer(strategy='median').fit(RAW[tr_idx])
    return np.hstack([a, ri.transform(RAW[tr_idx])]).astype(np.float32), np.hstack([b, ri.transform(ev_RAW)]).astype(np.float32)

KW = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)
def fit_predict(tr_idx, ev_X, ev_RAW, y):
    Xt, Xe = feats(tr_idx, ev_X, ev_RAW)
    return TabPFNRegressor(**KW).fit(Xt, y[tr_idx]).predict(Xe)

# smoke test: 90 rows, one fit, every shape printed, fails loudly before the long run
y0 = meta['CYP3A4_pIC50_direct_inhibition'].to_numpy(float); ok = np.flatnonzero(np.isfinite(y0))[:90]
Xt, Xe = feats(ok[:70], EMB[ok[70:]], RAW[ok[70:]])
p = TabPFNRegressor(**KW).fit(Xt, y0[ok[:70]]).predict(Xe)
print('smoke test ok: train', Xt.shape, 'eval', Xe.shape, 'pred', p.shape, 'finite', bool(np.isfinite(p).all()))

def save_atomic(path, **kw):
    tmp = Path(str(path) + '.tmp.npz'); np.savez_compressed(tmp, **kw); os.replace(tmp, path)

RESULTS = []
t_all = time.time()
for iso in ISO:
    final = OUT / f'{TAG}_{iso}.npz'
    y = meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
    obs = np.isfinite(y); dev = np.flatnonzero(obs & ~hold); ho = np.flatnonzero(obs & hold); allr = np.flatnonzero(obs)
    if final.exists():
        z = np.load(final); oof, hp = z['oof'], z['hold']; print(f'{iso}: finished earlier, skipping', flush=True)
    else:
        part = OUT / f'part_{TAG}_{iso}.npz'
        oof = np.full(len(dev), np.nan); hp = np.full(len(ho), np.nan); tp = np.full(len(EMB_TE), np.nan); done = []
        if part.exists():
            z = np.load(part, allow_pickle=True); oof, hp, tp = z['oof'], z['hp'], z['tp']; done = list(z['done'])
            print(f'{iso}: resuming, finished stages {done}', flush=True)
        def ckpt(): save_atomic(part, oof=oof, hp=hp, tp=tp, done=np.array(done))
        t0 = time.time()
        for k, (tr, va) in enumerate(GroupKFold(5).split(dev, dev, G[dev])):
            if f'f{k}' in done: continue
            oof[va] = fit_predict(dev[tr], EMB[dev[va]], RAW[dev[va]], y)
            done.append(f'f{k}'); ckpt(); print(f'  {iso} fold {k} ({time.time()-t0:.0f}s)', flush=True)
        if 'hold' not in done:
            hp = fit_predict(dev, EMB[ho], RAW[ho], y); done.append('hold'); ckpt(); print(f'  {iso} holdout model ({time.time()-t0:.0f}s)', flush=True)
        if 'test' not in done:
            tp = fit_predict(allr, EMB_TE, RAW_TE, y); done.append('test'); ckpt(); print(f'  {iso} test model ({time.time()-t0:.0f}s)', flush=True)
        assert np.isfinite(oof).all() and np.isfinite(hp).all() and np.isfinite(tp).all()
        save_atomic(final, oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
        part.unlink(missing_ok=True)
    d = float(spearmanr(y[dev], oof).statistic); h = float(spearmanr(y[ho], hp).statistic)
    RESULTS.append(dict(tag=TAG, iso=iso, dev_spearman=d, holdout_spearman=h, ref_tabqhts=REF[iso]))
    json.dump(RESULTS, open(OUT / f'results_{TAG}.json', 'w'), indent=2)
    print(f'{iso}: dev {d:.4f} (tabqhts {REF[iso]:.4f}) | holdout {h:.4f}', flush=True)
r = pd.DataFrame(RESULTS)
print('\nmacro dev', round(r.dev_spearman.mean(), 4), '| tabqhts macro', round(np.mean(list(REF.values())), 4),
      '| total time', round((time.time() - t_all) / 60, 1), 'min')


In [ ]:
import shutil
shutil.make_archive(f'/content/{TAG}_results', 'zip', OUT)
try:
    from google.colab import files
    files.download(f'/content/{TAG}_results.zip')
except Exception as e:
    print('download skipped:', type(e).__name__, '- the results are already on Drive in', OUT)